# S31. Estimação Estatística
## Statistical Estimation

[◀ Anterior](S30_Distribuicao_T_Student.ipynb) | [📖 Índice](S00_Index_Estatistica.ipynb) | [Próximo ▶](S32_Estimacao_Proporcao.ipynb)

## 🎯 Objetivos de Aprendizagem

- Diferenciar estimação pontual de estimação intervalar
- Compreender o conceito de estimador não-viesado
- Calcular e interpretar o erro padrão
- Construir e interpretar intervalos de confiança
- Entender o significado do nível de confiança

## 📝 Introdução

Quando queremos conhecer uma característica de uma população (média, proporção, variância), raramente podemos medir todos os elementos. Precisamos **estimar** esses valores usando amostras.

A **estimação estatística** é o processo de usar dados amostrais para fazer inferências sobre os parâmetros populacionais. Existem duas abordagens complementares: a **estimação pontual**, que dá um único valor, e a **estimação intervalar**, que fornece um intervalo de valores plausíveis com um nível de confiança associado.

## 📚 Explicação Detalhada

### 1. Estimação Pontual vs Intervalar

- **Estimação Pontual**: Um único valor calculado da amostra para estimar o parâmetro populacional. Exemplo: x̄ = 25.3 anos estima μ.
- **Estimação Intervalar**: Um intervalo [L, U] que, com certo nível de confiança, contém o parâmetro. Exemplo: IC 95% = [23.8, 26.8] anos.

### 2. Estimador Não-Viesado

Um estimador é **não-viesado** (ou não tendencioso) se seu valor esperado é igual ao parâmetro populacional:
- A média amostral x̄ é não-viesada para μ: E[x̄] = μ
- A variância amostral s² (com n-1) é não-viesada para σ²

### 3. Erro Padrão (Standard Error)

O erro padrão mede a precisão de uma estimativa — é o desvio padrão da distribuição amostral do estimador:

$$EP(\bar{x}) = \frac{\sigma}{\sqrt{n}}$$

Quando σ é desconhecido, usamos:

$$EP(\bar{x}) = \frac{s}{\sqrt{n}}$$

### 4. Intervalo de Confiança (IC)

Um intervalo de confiança de (1 - α) × 100% para a média é dado por:

$$\bar{x} \pm t_{\alpha/2, n-1} \cdot \frac{s}{\sqrt{n}}$$

### 5. Interpretação Correta do IC

**Correta**: Se construíssemos 100 ICs diferentes (de 100 amostras diferentes), esperaríamos que 95 deles contivessem o verdadeiro parâmetro.

**Incorreta**: "Há 95% de chance de μ estar dentro deste IC específico." (O parâmetro é fixo, o intervalo é aleatório.)

In [ ]:
# Importando bibliotecas
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print("Bibliotecas importadas com sucesso!")

## 💻 Exemplos Práticos

In [ ]:
# Exemplo 1: Estimação pontual da média
# Simulamos uma população e calculamos a estimativa pontual

np.random.seed(42)

# População: salários mensais (R$) com distribuição assimétrica
populacao = np.random.exponential(scale=3000, size=50000) + 1500
media_real = np.mean(populacao)

# Coletamos uma amostra
amostra = np.random.choice(populacao, size=100, replace=False)
media_amostral = np.mean(amostra)
desvio_amostral = np.std(amostra, ddof=1)
erro_padrao = desvio_amostral / np.sqrt(len(amostra))

print("=== Estimação Pontual ===")
print(f"Média populacional real (μ): R$ {media_real:.2f}")
print(f"Estimativa pontual (x̄): R$ {media_amostral:.2f}")
print(f"Erro de estimação: R$ {abs(media_real - media_amostral):.2f}")
print(f"Erro padrão: R$ {erro_padrao:.2f}")

In [ ]:
# Exemplo 2: Construindo um Intervalo de Confiança
# IC de 95% para a média de salários

n = len(amostra)
gl = n - 1
t_critico = stats.t.ppf(0.975, df=gl)
margem = t_critico * erro_padrao

ic_inf = media_amostral - margem
ic_sup = media_amostral + margem

print("=== Intervalo de Confiança (95%) ===")
print(f"Média amostral: R$ {media_amostral:.2f}")
print(f"n = {n}, gl = {gl}")
print(f"Valor crítico t: {t_critico:.4f}")
print(f"Margem de erro: R$ {margem:.2f}")
print(f"IC 95%: [R$ {ic_inf:.2f}, R$ {ic_sup:.2f}]")
print(f"Média real: R$ {media_real:.2f}")
print(f"A média real está dentro do IC? {ic_inf <= media_real <= ic_sup}")

In [ ]:
# Exemplo 3: DEMONSTRAÇÃO CRÍTICA — Interpretação do IC
# Construímos 100 ICs e verificamos quantos contêm a média real

np.random.seed(123)

media_verdadeira = 5000
desvio_verdadeiro = 800
n = 50
n_ics = 100

ic_inferiores = []
ic_superiores = []
contem_media = []

for _ in range(n_ics):
    amostra_sim = np.random.normal(media_verdadeira, desvio_verdadeiro, n)
    media_sim = np.mean(amostra_sim)
    s_sim = np.std(amostra_sim, ddof=1)
    ep_sim = s_sim / np.sqrt(n)
    t_crit_sim = stats.t.ppf(0.975, n - 1)
    margem_sim = t_crit_sim * ep_sim
    
    inf = media_sim - margem_sim
    sup = media_sim + margem_sim
    ic_inferiores.append(inf)
    ic_superiores.append(sup)
    contem_media.append(inf <= media_verdadeira <= sup)

proporcao = sum(contem_media) / n_ics

print("=== Interpretação do IC — Simulação ===")
print(f"Construímos {n_ics} ICs de 95%")
print(f"ICs que contêm a média real: {sum(contem_media)} de {n_ics}")
print(f"Proporção: {proporcao:.1%}")
print(f"Esperado: ~95%")

# Visualização
cores = ['green' if contem else 'red' for contem in contem_media]

plt.figure(figsize=(14, 5))
for i in range(n_ics):
    plt.plot([ic_inferiores[i], ic_superiores[i]], [i, i], color=cores[i], lw=0.8)

plt.axvline(media_verdadeira, color='blue', linestyle='--', 
            label=f'Média real = {media_verdadeira}', lw=2)
plt.xlabel('Valor', fontsize=12)
plt.ylabel('IC #', fontsize=12)
plt.title(f'{n_ics} Intervalos de Confiança de 95% — {int(proporcao*100)}% contêm a média real', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Exemplo 4: Efeito do nível de confiança na largura do IC
# Quanto maior a confiança, maior (mais largo) o intervalo

niveis = [0.80, 0.85, 0.90, 0.95, 0.99, 0.999]

print("=== Efeito do Nível de Confiança ===")
print(f"{'Nível':<12} {'t crítico':<12} {'Margem':<12} {'IC'}")
print("-" * 50)

for nivel in niveis:
    alpha = 1 - nivel
    t_crit = stats.t.ppf(1 - alpha/2, df=n-1)
    margem_n = t_crit * erro_padrao
    print(f"{nivel:<10.0%}  {t_crit:<12.4f} R$ {margem_n:<8.2f} [R$ {media_amostral - margem_n:.2f}, R$ {media_amostral + margem_n:.2f}]")

In [ ]:
# Exemplo 5: Cálculo direto do IC com scipy.stats.t.interval
# Método mais prático

ic_95 = stats.t.interval(confidence=0.95, df=n-1, loc=media_amostral, scale=erro_padrao)
ic_90 = stats.t.interval(confidence=0.90, df=n-1, loc=media_amostral, scale=erro_padrao)
ic_99 = stats.t.interval(confidence=0.99, df=n-1, loc=media_amostral, scale=erro_padrao)

print("=== Usando stats.t.interval ===")
print(f"IC 90%: [{ic_90[0]:.2f}, {ic_90[1]:.2f}]")
print(f"IC 95%: [{ic_95[0]:.2f}, {ic_95[1]:.2f}]")
print(f"IC 99%: [{ic_99[0]:.2f}, {ic_99[1]:.2f}]")

In [ ]:
# Exemplo 6: Verificando o viés do estimador x̄ via simulação
# Demonstramos que E[x̄] = μ (estimador não-viesado)

np.random.seed(42)

media_pop = 100
desvio_pop = 15
n_rep = 50000
n_amostra = 30

medias_amostrais_repetidas = []

for _ in range(n_rep):
    amostra_rep = np.random.normal(media_pop, desvio_pop, n_amostra)
    medias_amostrais_repetidas.append(np.mean(amostra_rep))

media_das_medias = np.mean(medias_amostrais_repetidas)

print("=== Verificação de Viés do Estimador ===")
print(f"Média populacional (μ): {media_pop}")
print(f"Média de {n_rep} estimativas de x̄: {media_das_medias:.4f}")
print(f"Diferença (viés): {abs(media_pop - media_das_medias):.4f}")
print(f"→ x̄ é não-viesado (E[x̄] ≈ μ)")

## ✅ Exercícios Resolvidos

**Exercício 1:** Uma amostra de 25 pacientes teve pressão sistólica média de 128 mmHg com desvio padrão de 10 mmHg. Calcule o IC de 95% para a média populacional.

In [ ]:
# Solução Exercício 1
n = 25
media = 128
s = 10
gl = n - 1

ep = s / np.sqrt(n)
ic = stats.t.interval(confidence=0.95, df=gl, loc=media, scale=ep)

print(f"n = {n}, média = {media}, s = {s}")
print(f"Erro padrão: {ep:.2f}")
print(f"IC 95% para pressão média: [{ic[0]:.2f}, {ic[1]:.2f}] mmHg")
print(f"\nInterpretação: Estamos 95% confiantes de que a pressão sistólica
média populacional está entre {ic[0]:.2f} e {ic[1]:.2f} mmHg.")

**Exercício 2:** Explique por que o IC de 99% é mais largo que o IC de 95%.

In [ ]:
# Solução Exercício 2 — demonstração
n = 25
media = 128
s = 10
ep = s / np.sqrt(n)

ic_95 = stats.t.interval(0.95, n-1, media, ep)
ic_99 = stats.t.interval(0.99, n-1, media, ep)

largura_95 = ic_95[1] - ic_95[0]
largura_99 = ic_99[1] - ic_99[0]

print("IC 95%:", [round(v, 2) for v in ic_95], f"→ largura: {largura_95:.2f}")
print("IC 99%:", [round(v, 2) for v in ic_99], f"→ largura: {largura_99:.2f}")
print(f"\nPara ter mais confiança (99% vs 95%), precisamos de um intervalo
mais largo. Há um trade-off entre precisão e confiança.")

## 📋 Exercícios Propostos

| Nível | Exercício |
|-------|-----------|
| 🟢 Fácil | 1. Uma amostra de n=40 tem média 72 e s=12. Calcule o erro padrão. |
| 🟡 Médio | 2. Construa o IC de 95% e 90% para os dados do exercício 1. Compare as larguras. |
| 🔴 Difícil | 3. Usando simulação, demonstre que 95% dos ICs de 95% contêm a média real. Use n=20, 1000 repetições, população normal. |

In [ ]:
# 🟢 Exercício 1: Escreva sua solução aqui
import numpy as np

# Sua solução


In [ ]:
# 🟡 Exercício 2: Escreva sua solução aqui
from scipy import stats

# Sua solução


## 🏆 Desafios

1. Pesquise sobre o conceito de "estimador de máxima verossimilhança" (MLE) e dê um exemplo.
2. Implemente uma função que calcule o tamanho de amostra necessário para uma margem de erro desejada.
3. Explique o trade-off entre viés e variância na estimação estatística.

## 📌 Resumo

- **Estimação pontual**: um único valor (x̄ para μ, s para σ).
- **Estimação intervalar**: intervalo com nível de confiança.
- **Erro padrão**: $\sigma / \sqrt{n}$ ou $s / \sqrt{n}$.
- **IC para média**: $\bar{x} \pm t_{\alpha/2, n-1} \times EP$.
- **Interpretação**: 95% dos ICs construídos conterão o parâmetro verdadeiro.

## 💡 Curiosidades

O conceito de intervalo de confiança foi introduzido por **Jerzy Neyman** em 1937. Antes dele, os estatísticos usavam apenas estimação pontual. Neyman mostrou que relatar apenas um valor sem indicar sua precisão podia ser enganoso. Sua abordagem revolucionou a estatística aplicada e hoje é padrão em todas as áreas científicas.

## ✅ Boas Práticas

1. Sempre reporte o IC junto com a estimativa pontual.
2. Use a interpretação correta: o IC contém ou não o parâmetro (não é probabilidade).
3. Escolha o nível de confiança baseado no contexto (95% é padrão, 99% para situações críticas).
4. Verifique as condições de aplicação (normalidade, independência, amostra aleatória).
5. Quanto maior a amostra, mais preciso o IC (menor margem de erro).

## ⚠️ Erros Comuns

| Erro | Como Evitar |
|------|-------------|
| "Há 95% de chance de μ estar neste IC" | O parâmetro é fixo — a probabilidade está no processo, não no intervalo. |
| Confundir erro padrão com desvio padrão | EP = s/√n, não s. |
| Usar Z quando deveria usar t (σ desconhecido) | Com σ desconhecido, use t de Student. |
| Ignorar as condições de aplicação | Amostra precisa ser aleatória e independente. |

## 📖 Referências

- [W3Schools — Confidence Intervals](https://www.w3schools.com/statistics/statistics_confidence_intervals.php)
- [Khan Academy — Confidence Intervals](https://pt.khanacademy.org/math/statistics-probability/confidence-intervals-one-sample)
- [SciPy — stats.t.interval](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.t.html)
- [Seeing Theory — Confidence Intervals (visual)](https://seeing-theory.brown.edu/frequentist-inference/index.html)

[◀ Anterior](S30_Distribuicao_T_Student.ipynb) | [📖 Índice](S00_Index_Estatistica.ipynb) | [Próximo ▶](S32_Estimacao_Proporcao.ipynb)

---
🧠 **Quer uma abordagem mais intuitiva?**
Visite o [companheiro de analogia: Estimacao Estatistica — O Chute no Escuro](S74_Estimacao_Analogias.ipynb)
para aprender este conteudo com uma historia do dia a dia.
